# 🏠 Linear Regression — House Price Predictor

### What is Linear Regression?
Think of it like drawing the **best straight line** through a scatter of data points.

For example:
> *"The bigger the house, the higher the price"*

Linear Regression finds the **mathematical formula** for that relationship so we can **predict** prices for houses we haven't seen yet.

---
### Our Goal
Use **Size, Bedrooms and Age** of a house to **predict its Price**.

```
Price = (Weight × Size) + (Weight × Bedrooms) + (Weight × Age) + Baseline
```

---
## Step 1 — Import Libraries

In [ ]:
import pandas as pd                              # for working with tables of data
import numpy as np                               # for numbers and math
import matplotlib.pyplot as plt                  # for charts

from sklearn.linear_model import LinearRegression        # the model
from sklearn.model_selection import train_test_split     # split data into train / test
from sklearn.metrics import mean_absolute_error, r2_score  # measure how good the model is

print("✅ Libraries loaded successfully!")

---
## Step 2 — Create the Dataset

We'll use a **synthetic (made-up but realistic) housing dataset** with 100 houses.

Each house has:
- **Size_sqft** — size of the house in square feet
- **Bedrooms** — number of bedrooms
- **Age_years** — how old the house is
- **Price** — the selling price (this is what we want to predict)

In [ ]:
np.random.seed(42)   # ensures we get the same random numbers every time
n = 100

size     = np.random.randint(800, 4000, n)    # house size between 800 and 4000 sqft
bedrooms = np.random.randint(1, 6, n)         # 1 to 5 bedrooms
age      = np.random.randint(1, 50, n)        # 1 to 50 years old

# Price is influenced by size, bedrooms and age + some random noise (real life is never perfect!)
price = (size * 150) + (bedrooms * 15000) - (age * 500) + np.random.normal(0, 20000, n)
price = np.round(price / 1000) * 1000         # round to nearest $1,000

df = pd.DataFrame({
    'Size_sqft' : size,
    'Bedrooms'  : bedrooms,
    'Age_years' : age,
    'Price'     : price
})

print(f"Dataset shape: {df.shape[0]} houses, {df.shape[1]} columns")

df.to_csv( 'Created_Housing_Data.csv', index_label=False )

df.head(10)

---
## Step 3 — Explore the Data

Before building any model, always look at your data first.

**`describe()`** gives us a quick statistical summary — min, max, average etc.

In [ ]:
df.describe().round(2)

### Visualise the Relationships
Let's plot each feature against Price to see if there's a pattern.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('How Each Feature Relates to House Price', fontsize=14, fontweight='bold')

features = ['Size_sqft', 'Bedrooms', 'Age_years']
colours  = ['steelblue', 'seagreen', 'tomato']
labels   = ['Size (sqft)', 'Number of Bedrooms', 'Age of House (years)']

for ax, feat, col, lab in zip(axes, features, colours, labels):
    ax.scatter(df[feat], df['Price'] / 1000, alpha=0.6, color=col, edgecolors='white', linewidth=0.5)
    ax.set_xlabel(lab, fontsize=11)
    ax.set_ylabel('Price ($000s)', fontsize=11)
    ax.set_title(f'{lab} vs Price', fontsize=11)
    ax.grid(True, linestyle='--', alpha=0.4)

plt.tight_layout()
plt.show()

print("💡 Observations:")
print("   ↑ Bigger size  → Higher price  (positive relationship)")
print("   ↑ More bedrooms → Higher price (positive relationship)")
print("   ↑ Older house  → Lower price   (negative relationship)")

---
## Step 4 — Split Data into Training and Testing Sets

We split our data into two groups:
- **Training set (80%)** — the model *learns* from this data
- **Testing set (20%)** — we *test* the model on data it has **never seen before**

This is like studying with a textbook (training) and then taking an exam with new questions (testing).

In [ ]:
# X = features (inputs)   |   y = target (what we want to predict)
X = df[['Size_sqft', 'Bedrooms', 'Age_years']]
y = df['Price']

# 80% training, 20% testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Total houses    : {len(df)}")
print(f"Training houses : {len(X_train)}  (80%) — model learns from these")
print(f"Testing houses  : {len(X_test)}   (20%) — model is tested on these")

---
## Step 5 — Train the Model

This is where the magic happens!

`model.fit()` tells the model to **look at the training data and learn the pattern**.

Behind the scenes it finds the best weights for:
```
Price = (w1 × Size) + (w2 × Bedrooms) + (w3 × Age) + Intercept
```

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)   # train the model

print("✅ Model trained!\n")
print("📐 What the model learned (coefficients / weights):")
print(f"   Every extra 1 sqft    adds  ${model.coef_[0]:,.0f}  to the price")
print(f"   Every extra bedroom   adds  ${model.coef_[1]:,.0f}  to the price")
print(f"   Every extra year old  takes ${abs(model.coef_[2]):,.0f}  off the price")
print(f"   Base price (intercept): ${model.intercept_:,.0f}")

---
## Step 6 — Make Predictions

Now we ask the model to predict prices for the **20 test houses it has never seen**.

In [ ]:
y_pred = model.predict(X_test)   # predict prices for test houses

# Side-by-side comparison: Actual vs Predicted
results = X_test.copy()
results['Actual Price']    = y_test.values
results['Predicted Price'] = np.round(y_pred / 1000) * 1000
results['Difference']      = (results['Predicted Price'] - results['Actual Price']).abs()

results = results.reset_index(drop=True)
results[['Size_sqft','Bedrooms','Age_years','Actual Price','Predicted Price','Difference']]

---
## Step 7 — Evaluate the Model

How good are our predictions? We use two metrics:

| Metric | What it means | Good value |
|--------|--------------|------------|
| **MAE** (Mean Absolute Error) | On average, how many dollars off are our predictions? | As low as possible |
| **R² Score** (R-Squared) | How much of the price variation does our model explain? | Close to 1.0 (100%) is perfect |

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
r2  = r2_score(y_test, y_pred)

print("📊 Model Performance on Test Data")
print("─" * 40)
print(f"   MAE (Average Error) : ${mae:,.0f}")
print(f"   R² Score            : {r2:.2%}")
print("─" * 40)
print(f"\n💡 Interpretation:")
print(f"   On average, our predictions are off by ${mae:,.0f}")
print(f"   Our model explains {r2:.1%} of house price variation")

---
## Step 8 — Visualise: Actual vs Predicted Prices

A perfect model would have all points sitting exactly on the diagonal line.
The closer the dots are to the line, the better our model is!

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Model Evaluation', fontsize=14, fontweight='bold')

# --- Chart 1: Actual vs Predicted ---
ax1 = axes[0]
ax1.scatter(y_test / 1000, y_pred / 1000, color='steelblue', alpha=0.7, edgecolors='white')

# Perfect prediction line
min_val = min(y_test.min(), y_pred.min()) / 1000
max_val = max(y_test.max(), y_pred.max()) / 1000
ax1.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Prediction')

ax1.set_xlabel('Actual Price ($000s)',    fontsize=11)
ax1.set_ylabel('Predicted Price ($000s)', fontsize=11)
ax1.set_title('Actual vs Predicted Price', fontsize=12)
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.4)
ax1.text(0.05, 0.95, f'R² = {r2:.2%}', transform=ax1.transAxes,
         fontsize=11, verticalalignment='top',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# --- Chart 2: Prediction Errors (Residuals) ---
ax2 = axes[1]
errors = y_test.values - y_pred
ax2.bar(range(len(errors)), errors / 1000,
        color=['tomato' if e < 0 else 'seagreen' for e in errors], alpha=0.7)
ax2.axhline(0, color='black', linewidth=1.2, linestyle='--')
ax2.set_xlabel('House (Test Set)',  fontsize=11)
ax2.set_ylabel('Error ($000s)',     fontsize=11)
ax2.set_title('Prediction Errors per House\n(Green = overestimated, Red = underestimated)', fontsize=11)
ax2.grid(True, linestyle='--', alpha=0.4, axis='y')

plt.tight_layout()
plt.show()

---
## Step 9 — Predict a Brand New House 🏡

Let's use the model to predict the price of a house that was **not in our dataset at all**.

In [ ]:
# Define a new house — change these values and re-run!
new_house = pd.DataFrame({
    'Size_sqft' : [2500],   # 2500 square feet
    'Bedrooms'  : [3],      # 3 bedrooms
    'Age_years' : [10],     # 10 years old
})

predicted_price = model.predict(new_house)[0]

print("🏡 New House Details")
print("─" * 35)
print(f"   Size     : {new_house['Size_sqft'][0]:,} sqft")
print(f"   Bedrooms : {new_house['Bedrooms'][0]}")
print(f"   Age      : {new_house['Age_years'][0]} years")
print("─" * 35)
print(f"💰 Predicted Price: ${predicted_price:,.0f}")

---
## Summary — What We Did

```
1. LOAD DATA       → Created a dataset of 100 houses
         ↓
2. EXPLORE         → Visualised how size, bedrooms & age relate to price
         ↓
3. SPLIT           → 80% to train the model, 20% to test it
         ↓
4. TRAIN           → model.fit() — model learns the pattern
         ↓
5. PREDICT         → model.predict() — model guesses prices
         ↓
6. EVALUATE        → MAE tells us average error, R² tells us accuracy
         ↓
7. PREDICT NEW     → Fed in a brand new house → got a price estimate
```

| Concept | Plain English |
|---------|---------------|
| **Feature (X)** | The inputs we use to make a prediction (size, bedrooms, age) |
| **Target (y)** | What we are trying to predict (price) |
| **Coefficient** | How much each feature affects the price |
| **Intercept** | The base price even before any features are considered |
| **MAE** | Average dollar amount our predictions are off by |
| **R² Score** | % of price variation our model can explain (higher = better) |